## Unit Test Cheatsheet 

This notebook provides a reference for writing unit tests in Python using unnitest (and pytest), specifically tailored for data science and AI workflows. 

### What is a unit test 
- Short, isolated test that verifies whether a single "unit" of code (usually a function or method) behaves as expected
- We will define the expected input and output, then run the test to confirm that my code does what it is supposed to do. 
- If it fails, the test fails and flags the issue


### Basics of unit testing
- When we write class TestMath(unittest.TestCase), we are telling Python that this class contains some tests. 
    - When run, Python executes all methods that start with test_ (e.g. test_add)
        - Behind the scenes, Python's unittest framework creates an instance of TestMath automatically 
        - It then calls the method like test_add() and handles the set-up/teardown logic itself 
            - I.e. the unittest framework automatically instantiates test class for each test method. I.e. it is doing 
                - test_case = TestMath(methodName="test_add")
                - test_case.test_add()
        - Use self.assertEqual() instead of plain assert because self is the instance of the test case, and assertEqual is the method it inherits
    - Then the if __name__ = "__main__": unittest.main() means if run the file directly (e.g. typing file_name.py in terminal), then run all tests in it
        - unittest.main() scans the file for classes that inherit from unittest.TestCase, looks for all methods that start with test_ and runs them 
- NOTE: unittest.main() is designed for .py files and executed directly 
    - In Jupyter notebooks or IPython environments, extra command-line arguments can confuse it and cause errors 
    - To safely run unit tests in notebooks, use "unittest.main(argv=['first-arg-is-ignored'], exit=False)"
        - argv stands for argument vector, a list of command line arguments passed to the Python script 
            - Normally, Python scripts get their argv list from sys.argv, which contains things like ['script_name'.py, "--some-flag"]
            - unittest.main() also expects first item in argv to be the script name, so we feed it something harmless
                - Jupyter passes extra flags that unittest dosen't understand, causing it to crash 
                - Hence, we override argv manually with a fake value (here its ['first-arg-is-ignored']) to keep unittest happy
- False is a useful Boolean keyword and is one of only two valid Boolean values: True and False 

In [4]:
#Testing basic functions
import unittest 

def add(a, b): 
    return a + b 

class TestMath(unittest.TestCase): #inherit from the unittest.TestCase, which handles the set-up behind the scenes. Hence don't need to write my own __init__()
    def test_add(self): #self here refers to the TestMath instance
        self.assertEqual(add(2,3), 5)
        self.assertEqual(add(-1,1), 0)
        self.assertNotEqual(add(2,2), 5)
    
    def test_type(self): 
        self.assertIsInstance(add(1,1), int)

unittest.main(argv=['Red_herring'], exit=False)

..
----------------------------------------------------------------------
Ran 2 tests in 0.001s

OK


In [19]:
#Testing Data Cleaning / Transformation 
import pandas as pd

def scale_features(df): #recall that by definition, pandas calculate mean() and std() along columns (axis=0) corresponding to each feature 
    return (df - df.mean()) / df.std() #this operates element-wise, and uses broadcasting under the hood. df.mean() creates a row of means, then Pandas broadcasts this row across all the rows 
    #note that pandas broadcasting uses Numpy under the hood (dimensions must match or be 1, if one has less dims prepend 1), but pandas is smarter as it aligns by labels first (column headers if working across columns, index labels if working across rows), not just shapes

class TestProcessing(unittest.TestCase): 
    def setUp(self): #do this to just test if the uniitest are working. But if not, then have to define df outside of this for Python to resolve
        self.df = pd.DataFrame([[1,2], [3,4]])

    def test_shape(self): 
        self.assertEqual(scale_features(self.df).shape, self.df.shape) #after scaling, df must have same shape as the original (this normalisation is shape preserving transformation)
    
    def test_no_nulls(self):
        self.assertFalse(scale_features(self.df).isnull().values.any()) #assertFalse is a built-in assertion method that checks whether expression evaluates to False. Fails if expression is True

unittest.main(argv=['Red_herring'], exit=False)

....
----------------------------------------------------------------------
Ran 4 tests in 0.006s

OK


In [20]:
#Test Feature Engineering 
import pandas as pd

def extract_year(date_series): #note: date_series is a column or Series of timestamps, and .dt is a datetime accessor in Pandas -> unlocks date-time specific attributes (e.g. year, month, weekday). dt.year pulls out year from each date
    return date_series.dt.year

class TestFeatures(unittest.TestCase): 
    def test_year_extraction(self): 
        test_dates = pd.Series(pd.to_datetime(['2021-02-02', '2020-05-01'])) #pd.to_datetime() converts strings to datetime objects (DatetimeIndex, BUT NOT YET A SERIES), where each string becomes a Timestamp. When wrap it with pd.Series(), convert into a Pandas Series of Timestaps
        expected = pd.Series([2021, 2020]) #these must be integers and not strings because extract_year() returns integers 
        pd.testing.assert_series_equal(extract_year(test_dates), expected) #this is the pandas native way to compare two series deeply

In [ ]:
#Testing ML Model Predictions 

class TestModel(unittest.TestCase): 
    def test_prediction_shape(self): 
        preds = model.predict(X_test) #assumes there's some model and X_test object already defined
        self.assertEqual(preds.shape[0], X_test.shape[0]) #checks that no. predictions equals the no. input samples. preds.shape returns a tuple representing the dimensions of the output array, and .shape[0] represents the first dimension which is the no. rows (samples)
    
    def test_prediction_range(self): 
        preds = model.predict_proba(X_test)[:,1] #model.predict_proba() returns a 2D NumPY array of shape (n_samples, n_classes). Here we grab just the second column, i.e. probability of class 1 for each sample
        self.assertTrue(((preds >= 0) & (preds <=1)).all()) #note that and is used for single Boolean values, whereas if we want element-wise logic we use &. This will perform element-wise comparison and returns a Boolean array, and .all() checks if every value is True

In [ ]:
#Only run the below in a .py file; running in ipynb throws up errors because of flags
if __name__ == '__main__': 
    unittest.main()